In [175]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

In [176]:
df = sns.load_dataset("diamonds")
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [177]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=0)

categoric_transformer = OneHotEncoder()
numeric_transformer = StandardScaler()
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']
categorical_features = ['cut', 'color', 'clarity']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_predict = pipeline.predict(X_test)

score = pipeline.score(X_test, y_test)

print(f'Pipeline Score: {score}')
print(f"Mean Squared Error: {mean_squared_error(y_test, y_predict)}")
print(f"R2 Score: {r2_score(y_test, y_predict)}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_predict)}")
print(f"Mean Absolute Percentage Error: {mean_absolute_percentage_error(y_test, y_predict)}")
print(f"Root Mean Squared Error: {root_mean_squared_error(y_test, y_predict)}")

Pipeline Score: 0.9212394719973422
Mean Squared Error: 1248486.7135639724
R2 Score: 0.9212394719973422
Mean Absolute Error: 737.4354677843595
Mean Absolute Percentage Error: 0.3940861946973969
Root Mean Squared Error: 1117.3570215307068


In [178]:
row = X_test.head(1).copy()

print(pipeline.predict(row))

[4915.63277945]


In [179]:
custom_input = {
    'carat': [0.5],
    'depth': [61.5],
    'table': [55.0],
    'x': [5.1],
    'y': [5.15],
    'z': [3.98],
    'cut': ['Ideal'],
    'color': ['E'],
    'clarity': ['VS1']
}

# Convert to DataFrame (pipeline expects a DataFrame, not a dict)
custom_row = pd.DataFrame(custom_input)

# Predict
predicted_price = pipeline.predict(custom_row)
print(f"Predicted Price: ${predicted_price[0]:,.2f}")

Predicted Price: $2,352.75


In [180]:
print(df['cut'].unique())
print(df['color'].unique())
print(df['clarity'].unique())

['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
Categories (5, object): ['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']
['E', 'I', 'J', 'H', 'F', 'G', 'D']
Categories (7, object): ['D', 'E', 'F', 'G', 'H', 'I', 'J']
['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']
Categories (8, object): ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']


In [182]:
import pickle
import os

if not os.path.exists("saved_models"):
    os.mkdir("saved_models")

with open('saved_models/diamonds_price_model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)
    
print("Model Saved Successfully!")

Model Saved Successfully!


In [183]:
with open('saved_models/diamonds_price_model.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)
    
print("Loaded Successfully!")

Loaded Successfully!


In [184]:
custom_input = {
    'carat': [0.5],
    'depth': [61.5],
    'table': [55.0],
    'x': [5.1],
    'y': [5.15],
    'z': [3.98],
    'cut': ['Ideal'],
    'color': ['E'],
    'clarity': ['VS1']
}

# Convert to DataFrame (pipeline expects a DataFrame, not a dict)
custom_row = pd.DataFrame(custom_input)

# Predict
predicted_price = loaded_pipeline.predict(custom_row)
print(f"Predicted Price: ${predicted_price[0]:,.2f}")

Predicted Price: $2,352.75
